<a href="https://colab.research.google.com/github/PadawanXXVI/kmeans-3d-cluster/blob/main/distribuicao-renda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Projeto de Clusterização — Método IDEB Adaptado
Notebook estruturado para análises de clusterização utilizando o método aplicado ao IDEB, adaptado para qualquer base de dados carregada pelo usuário.

## 🔹 Seção 1: Importações e Setup


In [ ]:
# ================================================
# IMPORTAÇÕES BÁSICAS
# ================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import pickle
import requests
from io import StringIO

# ================================================
# CONFIGURAÇÕES GERAIS
# ================================================
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
sns.set(style="whitegrid")

# ================================================
# FUNÇÃO PARA CARREGAR CSV DO GITHUB (RAW)
# ================================================
def load_from_github(raw_url: str) -> pd.DataFrame:
    """Carrega CSV publicado no GitHub (raw URL)."""
    response = requests.get(raw_url)
    if response.status_code != 200:
        raise ValueError("Erro ao acessar o arquivo. Verifique a URL RAW.")
    # Tentando ler com delimitador ';', comum em CSVs em algumas regiões
    # e usando on_bad_lines='warn' para não falhar completamente em linhas problemáticas.
    return pd.read_csv(StringIO(response.text), sep=';', on_bad_lines='warn')


## 🔹 Seção 2: Carregamento da Base
Insira a URL RAW do GitHub ou faça upload manual do CSV.


In [ ]:
# OPÇÃO 1 — Carregar usando URL RAW
raw_url = "https://raw.githubusercontent.com/PadawanXXVI/kmeans-3d-cluster/main/distribuicao-renda.csv"
if raw_url and raw_url != "YOUR_GITHUB_RAW_URL_HERE":
    df = load_from_github(raw_url)
else:
    # OPÇÃO 2 — Upload manual
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])

# Visualização inicial
display(df.head())
print(df.shape)
df.info()
df.describe(include="all")

## 🔹 Seção 3: Pré-processamento
Aqui tratamos nulos, criamos novas features e selecionamos apenas colunas numéricas.


In [ ]:
# Remover duplicatas
df = df.drop_duplicates()

# Normalizar nomes de colunas
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# Identificar colunas numéricas
num_cols = df.select_dtypes(include=np.number).columns.tolist()

df_num = df[num_cols].copy()

# Tratar nulos (simples: dropna)
df_num = df_num.dropna()

print("Colunas numéricas selecionadas:", num_cols)
df_num.describe()


## 🔹 Seção 4: Padronização
Padronização das variáveis numéricas para clusterização.


In [ ]:
scaler = StandardScaler()
df_pad = scaler.fit_transform(df_num)

df_pad = pd.DataFrame(df_pad, columns=[col + "_PAD" for col in num_cols])

# Concatenar original + padronizado
df_cluster_base = pd.concat([df_num.reset_index(drop=True), df_pad], axis=1)

df_cluster_base.head()


## 🔹 Seção 5: Método do Cotovelo + Silhouette
Testamos K=1 a 20 para determinar o número ideal de clusters.


In [ ]:
wcss = []
sil_scores = []

X = df_pad.copy()
K_MAX = 20

for k in range(2, K_MAX + 1):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

# Plot Cotovelo
plt.figure(figsize=(10,5))
plt.plot(range(2, K_MAX+1), wcss, marker='o')
plt.title("Método do Cotovelo")
plt.xlabel("Número de clusters (k)")
plt.ylabel("WCSS")
plt.show()

# Plot Silhouette
plt.figure(figsize=(10,5))
plt.plot(range(2, K_MAX+1), sil_scores, marker='o', color='green')
plt.title("Coeficiente de Silhouette")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette")
plt.show()


## 🔹 Seção 6: Ajuste do Modelo Final
Aqui escolhemos o K, treinamos o KMeans e criamos os rótulos dos clusters.


In [ ]:
K_FINAL = 4  # defina após analisar os gráficos

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df_cluster_base["cluster"] = kmeans.fit_predict(X)

# Converter cluster numérico → A, B, C, D...
df_cluster_base["cluster"] = df_cluster_base["cluster"].apply(lambda x: chr(65 + x))

# Criar rótulos interpretáveis
cluster_labels = {
    "A": "Perfil A",
    "B": "Perfil B",
    "C": "Perfil C",
    "D": "Perfil D",
}

df_cluster_base["DS_cluster"] = df_cluster_base["cluster"].map(cluster_labels)

df_cluster_base.head()


## 🔹 Seção 7: Visualizações Avançadas
Inclui scatter 3D, KDE, boxplots, spider charts e gráficos comparativos.


In [ ]:
fig = px.scatter_3d(
    df_cluster_base,
    x=num_cols[0],
    y=num_cols[1],
    z=num_cols[2] if len(num_cols) > 2 else num_cols[0],
    color="DS_cluster",
    symbol="cluster",
)
fig.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.kdeplot(
    data=df_cluster_base,
    x=num_cols[0],
    hue="DS_cluster",
    fill=True
)
plt.title("Densidade por Cluster")
plt.show()


In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df_cluster_base[num_cols])
plt.title("Boxplots das Variáveis Originais")
plt.xticks(rotation=45)
plt.show()


In [ ]:
medianas = df_cluster_base.groupby("DS_cluster")[num_cols].median()

categories = medianas.columns.tolist()
N = len(categories)

angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

plt.figure(figsize=(8,8))
for label, row in medianas.iterrows():
    values = row.tolist()
    values += values[:1]
    plt.polar(angles, values, marker='o', label=label)

plt.title("Spider Chart — Medianas por Cluster")
plt.legend()
plt.show()


In [ ]:
value_counts = df_cluster_base["DS_cluster"].value_counts()

plt.figure(figsize=(6,6))
plt.pie(value_counts, labels=value_counts.index, autopct="%1.1f%%")
plt.title("Distribuição dos Clusters")
plt.show()


## 🔹 Seção 8: Exportação de Resultados
Geramos CSV final, scalers e modelo KMeans para uso em produção.


In [ ]:
# Exportar CSV
df_cluster_base.to_csv("resultado_clusterizacao.csv", index=False)

# Salvar scaler e modelo
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("kmeans.pkl", "wb") as f:
    pickle.dump(kmeans, f)

print("Arquivos exportados:")
print(" - resultado_clusterizacao.csv")
print(" - scaler.pkl")
print(" - kmeans.pkl")
